In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy import ndimage

img_path = "/dataset/FLARE-MedFM/val/CT_Lesion/FLARE23Ts/"
our_res_path = "/dataset/FLARE-MedFM/outputs_submission_test/"

imgf = np.sort(os.listdir(img_path))
predf = np.sort(os.listdir(our_res_path))

imgs = []
for img_file in imgf:
    if img_file.endswith(".npz"):
        gt = np.load(os.path.join(img_path, img_file))
        imgs.append(gt)
our_preds = []
for gt_file in predf:
    if gt_file.endswith(".nii.gz"):
        dat = nib.load(os.path.join(our_res_path, gt_file))
        our_preds.append(dat.get_fdata())


In [ ]:
# ==== Config (edit here) =====================================================
CMAP_NAME        = "gist_rainbow"   # Colormap for overlays
ALPHA            = 0.5          # Overlay opacity
CROP_THRESHOLD   = 0.05        # Percent of cumulative intensity to keep when cropping
CROP_MARGIN_PX   = 10           # Extra pixels to include after cropping
CROP_MARGIN_PXT  = 90
PERCENTILES      = [0.05, 0.25, 0.5, 0.75, 0.95]  # Which ranks to display
RANK_ASCENDING   = False         # True: worst → best by Dice; False: best → worst
TEXT_Y_OFFSET    = 20           # Pixels from top of each row for overlay text
TEXT_COLOR       = "white"     # Overlay text color
TEXT_BG          = "black"      # Background box color
TEXT_FONTSIZE    = 14
# ============================================================================

def macro_dice_per_file(gt_volume: np.ndarray, pred_volume: np.ndarray) -> float:
    labels = np.unique(gt_volume)
    labels = labels[labels != 0]
    if labels.size == 0:
        return np.nan
    dices = []
    for lab in labels:
        g = (gt_volume == lab)
        p = (pred_volume == lab)
        inter = (g & p).sum()
        denom = g.sum() + p.sum()
        dices.append((2.0 * inter) / (denom + 1e-8))
    return float(np.mean(dices)) if dices else np.nan

def plot_overlays_side_by_side(dat, pred, cmap_name=CMAP_NAME, alpha=ALPHA,
                               crop_threshold=CROP_THRESHOLD, crop_margin_px=CROP_MARGIN_PX,
                               crop_margin_pxt=CROP_MARGIN_PXT):
    def get_gt_center_of_mass(gt, dim):
        try:
            return int(ndimage.center_of_mass(gt, labels=gt > 0, index=1)[dim])
        except Exception:
            return gt.shape[dim] // 2

    gt, im, recist = dat['gts'], dat['imgs'], dat['recist'].copy()
    c = get_gt_center_of_mass(recist==1, dim=0)
    vals = np.unique(gt[c])
    recist[~np.isin(recist, vals)] = 0
    pred[~np.isin(pred, vals)] = 0

    rec_slice, gt_slice, pred_slice, img_slice = recist.sum(0), gt[c], pred[c], im[c]

    def to_rgb(img):
        arr = img.astype(float)
        arr = (arr - arr.min()) / (arr.ptp() + 1e-8)
        arr = (arr * 255).astype(np.uint8)
        return np.stack([arr, arr, arr], axis=-1)

    base_rgb = to_rgb(img_slice)
    cmap = plt.colormaps[cmap_name]

    def overlay(base, mask, alpha=alpha):
        rgb = base.copy()
        if np.any(mask):
            norm_mask = mask.astype(float) / (mask.max() + 1e-8)
            colored = cmap(norm_mask)[..., :3]
            colored = (colored * 255).astype(np.uint8)
            mask_bin = mask > 0
            rgb[mask_bin] = (
                (1 - alpha) * rgb[mask_bin] + alpha * colored[mask_bin]
            ).astype(np.uint8)
        return rgb

    rec_img, gt_img, pred_img = overlay(base_rgb, rec_slice, alpha=1.0), overlay(base_rgb, gt_slice), overlay(base_rgb, pred_slice)

    def get_crop_params(rec_img):
        rec_img_ = rec_img.copy()
        rec_img_[rec_img > 50] = 0
        x_proj = np.sum(np.sum(rec_img_.astype(float), axis=2), axis=0)
        y_proj = np.sum(np.sum(rec_img_.astype(float), axis=2), axis=1)
        if x_proj.sum() == 0 or y_proj.sum() == 0:
            H, W = rec_img.shape[:2]
            return 0, W, 0, H
        x_cumsum = np.cumsum(x_proj) / x_proj.sum()
        y_cumsum = np.cumsum(y_proj) / y_proj.sum()
        x_start = np.argmax(x_cumsum > crop_threshold) - crop_margin_px
        x_end   = np.argmax(x_cumsum > (1 - crop_threshold)) + crop_margin_px
        y_start = np.argmax(y_cumsum > crop_threshold) - crop_margin_pxt
        y_end   = np.argmax(y_cumsum > (1 - crop_threshold)) + crop_margin_px
        return x_start, x_end, y_start, y_end

    xstart, xend, ystart, yend = get_crop_params(rec_img)
    H, W = rec_img.shape[:2]
    xstart, ystart = max(0, xstart), max(0, ystart)
    xend, yend = min(W, xend), min(H, yend)

    rec_img, gt_img, pred_img = rec_img[ystart:yend], gt_img[ystart:yend], pred_img[ystart:yend]
    return np.concatenate([rec_img, gt_img, pred_img], axis=1), c

def pick_indices_by_percentiles(scores, percentiles, ascending=True):
    scores = np.asarray(scores)
    filled = np.where(np.isnan(scores), -np.inf if ascending else np.inf, scores)
    order = np.argsort(filled)
    if not ascending:
        order = order[::-1]
    n = len(order)
    idxs = []
    for p in percentiles:
        p = float(np.clip(p, 0.0, 1.0))
        r = int(round(p * (n - 1))) if n > 1 else 0
        idxs.append((p, int(order[r])))
    return idxs



In [ ]:
macro_dices = [macro_dice_per_file(dat['gts'], pred) for dat, pred in zip(imgs, our_preds)]
chosen = pick_indices_by_percentiles(macro_dices, PERCENTILES, ascending=RANK_ASCENDING)

In [ ]:
rows, titles = [], []
for p, idx in chosen:
    comb, c = plot_overlays_side_by_side(imgs[idx], our_preds[idx])
    rows.append(comb)
    filename = (imgf[idx] if idx < len(imgf) else "N/A").replace(".npz", "")
    titles.append(f"{int(p*100)}th percentile, case {filename}, DSC={macro_dices[idx]:.3f}, slice {c}".replace("_", " "))

# Stack vertically
canvas = np.concatenate(rows, axis=0)

# Show single image and overlay text
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(canvas)
ax.axis("off")

# Overlay text at start of each row
y_offset = 0
for row_img, title in zip(rows, titles):
    ax.text(
        10, y_offset + TEXT_Y_OFFSET, title,
        color=TEXT_COLOR, fontsize=TEXT_FONTSIZE,
        bbox=dict(facecolor=TEXT_BG, alpha=0.5, pad=2),
        ha="left", va="top"
    )
    y_offset += row_img.shape[0]

# add text under the plot
subtext = ["Input Marker", "Ground Truth", "Lite ENSAM"]

x_offset = row_img.shape[1] / 7 + 40
for sub in subtext:
    ax.text(
        x_offset, row_img.shape[0] * (len(PERCENTILES)), sub,
        color="black", fontsize=TEXT_FONTSIZE,
        bbox=dict(facecolor="white", alpha=0.0, pad=2),
        ha="center", va="top"
    )
    x_offset += row_img.shape[1]//3
plt.tight_layout(pad=0)
plt.savefig("figs/png/overlay_plot.png", dpi=350, bbox_inches="tight")
plt.savefig("figs/pgf/overlay_plot.pgf", dpi=350, bbox_inches="tight")
plt.show()

In [ ]:
def get_input_output(dat, pred, cmap_name=CMAP_NAME, alpha=ALPHA):
    def get_gt_center_of_mass(gt, dim):
        try:
            return int(ndimage.center_of_mass(gt, labels=gt > 0, index=1)[dim])
        except Exception:
            return gt.shape[dim] // 2

    gt, im, recist = dat['gts'], dat['imgs'], dat['recist'].copy()
    c = get_gt_center_of_mass(recist==1, dim=0)
    vals = np.unique(gt[c])
    recist[~np.isin(recist, vals)] = 0
    pred[~np.isin(pred, vals)] = 0

    rec_slice, gt_slice, pred_slice, img_slice = recist.sum(0), gt[c], pred[c], im[c]

    def to_rgb(img):
        arr = img.astype(float)
        arr = (arr - arr.min()) / (arr.ptp() + 1e-8)
        arr = (arr * 255).astype(np.uint8)
        return np.stack([arr, arr, arr], axis=-1)

    base_rgb = to_rgb(img_slice)
    cmap = plt.colormaps[cmap_name]

    def overlay(base, mask, alpha=alpha):
        rgb = base.copy()
        if np.any(mask):
            norm_mask = mask.astype(float) / (mask.max() + 1e-8)
            colored = cmap(norm_mask)[..., :3]
            colored = (colored * 255).astype(np.uint8)
            mask_bin = mask > 0
            rgb[mask_bin] = (
                (1 - alpha) * rgb[mask_bin] + alpha * colored[mask_bin]
            ).astype(np.uint8)
        return rgb

    rec_img, gt_img, pred_img = overlay(base_rgb, rec_slice, alpha=1.0), overlay(base_rgb, gt_slice), overlay(base_rgb, pred_slice)
    return rec_img, pred_img

rec, pred = get_input_output(imgs[-1], our_preds[-1])

plt.imsave("figs/png/input.png", rec)
plt.imsave("figs/png/output.png", pred)